In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

In [ ]:
adatas = []
'''
adatas.append(sc.read_h5ad('reheatHeart/Simonson_2023.h5ad'))
adatas.append(sc.read_h5ad('reheatHeart/Kuppe_2022.h5ad'))
'''
for filename in os.listdir('reheatHeart'):
    file_path = os.path.join('reheatHeart', filename)
    adatas.append(sc.read_h5ad(file_path))

adatas[0]

In [ ]:
common_genes = set.intersection(*(set(adata.var_names) for adata in adatas))
len(common_genes)

In [ ]:
for i in range(len(adatas)):
    ad = adatas[i]
    ad = ad[~ad.obs['annotation_MOFA'].isna()]
    ad = ad[:, list(common_genes)]
    adatas[i] = ad
adatas[0]

In [ ]:
adata = an.concat(
    adatas,
    join='inner',       
)
adata

In [ ]:
adata = adata[adata.obs['cond_test']=='fibrosis']
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000, inplace=True)
adata = adata[:, adata.var['highly_variable']]
sc.pp.scale(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color=['study','annotation_MOFA'])

In [ ]:
sce.pp.harmony_integrate(adata, key='study')
adata

In [ ]:
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['study','annotation_MOFA'])

In [ ]:
#adata.write("reheatHeart/reheatHeart.h5ad")